# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [ ]:
# Import packages
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import warnings

warnings.filterwarnings("ignore")

tf.keras.backend.set_floatx("float64")

# Define random seed for whole notebook
RSEED = 42

In [ ]:
# Load data
df = pd.read_csv("../data/boston.csv")

# Define target
y = df.pop("MEDV")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [ ]:
# Scale numerical features
# Scale numerical values
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

In [ ]:
scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

> [!NOTE]
> This solution notebook lives in the `solutions/` folder. The model is created by notebook 3 (`day_2/03_overfit_underfit.ipynb`), which saves it to `day_2/saved_model/`, so the cells below reach it with the relative path `../day_2/saved_model/my_large_model.keras`.
>
> Run notebook 3 first so the model exists, otherwise these cells will not find it.

In [ ]:
!ls ../day_2/saved_model

In [ ]:
!ls ../day_2/saved_model/my_large_model.keras

## SavedModel format

Reload a fresh Keras model from the saved model:

In [ ]:
# Load the saved model
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("../day_2/saved_model/my_large_model.keras")

# Check its architecture
new_large_model.summary()

The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [ ]:
# Evaluate the restored model
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")

The Model MSE is higher than in the notebook before, even though we are using the same data-split. We are using the same model and the same data. Therefore, we would assume that we also receive the same MSE. 

> **Exercise:** Can you find out where this notebook varies from the procedure in the notebook before? It might help to have a look at the values in X_train in both notebooks.

<details><summary>
Click here for a hint...
</summary>
Check out how the data were preprocessed. Did both notebooks use the same scaler?
</details>

The model and the train/test split are identical to the previous notebook, so the only thing that can change the predictions is how the features were preprocessed. Let's pin down that difference and fix it.

### Step 1: Rebuild both scalers

The previous notebook (`03_overfit_underfit`) scaled the feature columns with a `StandardScaler`, while this notebook scaled the same columns with a `MinMaxScaler`. We reload the data with the same split and fit both scalers on the training columns, so we can compare them directly.

In [ ]:
from sklearn.preprocessing import StandardScaler

# reload and split exactly as before
df_cmp = pd.read_csv("../data/boston.csv")
y_cmp = df_cmp.pop("MEDV")
X_train_cmp, X_test_cmp, y_train_cmp, y_test_cmp = train_test_split(
    df_cmp, y_cmp, random_state=RSEED
)

# fit both scalers on the training columns only
minmax_scaler = MinMaxScaler().fit(X_train_cmp[col_scale])      
standard_scaler = StandardScaler().fit(X_train_cmp[col_scale])  # the scaler NOTEBOOK 3 used

### Step 2: Compare the scaled columns

The two scalers map the same columns onto very different value ranges. `MinMaxScaler` squeezes each column into 0 to 1, so every value is non-negative. `StandardScaler` centres each column on 0, so it also produces negative values. The model was trained on the second kind of input.

In [ ]:
minmax_values = minmax_scaler.transform(X_train_cmp[col_scale])
standard_values = standard_scaler.transform(X_train_cmp[col_scale])

print(f"MinMaxScaler   scaled columns range: {minmax_values.min():.2f} to {minmax_values.max():.2f}")
print(f"StandardScaler scaled columns range: {standard_values.min():.2f} to {standard_values.max():.2f}")

### Step 3: Re-evaluate the model with the matching scaler

If the scaler is the problem, scaling the test set with `StandardScaler` (exactly as notebook 3 did) and evaluating the same loaded model should bring the MSE back down to the notebook 3 value.

In [ ]:
# scale the test set the way notebook 3 did: StandardScaler on the same columns
X_test_std = X_test_cmp.copy()
X_test_std[col_scale] = standard_scaler.transform(X_test_cmp[col_scale])

# evaluate the same loaded model on the correctly scaled test set
with tf.device("/cpu:0"):
    loss_std, mse_std = new_large_model.evaluate(
        X_test_std.values, y_test_cmp.values, verbose=2
    )

print(f"MSE with MinMaxScaler (this notebook):        {mse:.2f}")
print(f"MSE with StandardScaler (matching notebook 3): {mse_std:.2f}")

### Answer

The two notebooks use a **different scaler**. Notebook 3 trained the model on features scaled with a `StandardScaler` (mean 0, standard deviation 1), but this notebook scaled the features with a `MinMaxScaler` (range 0 to 1). A neural network learns weights tuned to the exact scale of its training inputs, so feeding it data on a different scale shifts every prediction and inflates the MSE.

Scaling the test set with the same `StandardScaler` brings the MSE back down to the notebook 3 level, which confirms the scaler was the cause.

Preprocessing is part of the model. Whatever transformation you fit during training must be applied in exactly the same way at prediction time. In practice you would **save the fitted scaler** alongside the model (for example with `joblib`) and reload it, rather than fitting a new, different scaler.